### IMPORTACIONES

In [4]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio import Entrez
import pandas as pd
import json


### DATOS DE LAS SECUENCIAS

In [5]:
secuencias= SeqIO.parse("../data/raw/sequences.fasta", "fasta")

lista=[]
for sec in secuencias:
    lista.append({"ID":sec.id, "longitud": len(sec.seq), "Ns": sec.seq.count("N"), "registro": sec})

df_info= pd.DataFrame(lista)
df_info.describe()

,longitud,Ns
count,116.000000,116.0
mean,544.775862,0.0
std,425.658183,0.0
min,190.000000,0.0
25%,261.000000,0.0
50%,397.000000,0.0
75%,626.000000,0.0
max,1931.000000,0.0


### PERMISIVIDAD

Con estos min y max me quedan suficientes secuencias para seguir con el proceso

In [6]:
LONG_MIN = 900   
LONG_MAX = 1931

filtradas = df_info[(df_info["longitud"] >= LONG_MIN) & (df_info["longitud"] <= LONG_MAX)]
print(f"Secuencias conservadas: {len(filtradas)}")

Secuencias conservadas: 25


### REPETIDAS

In [30]:
seqs_unicas = []
vistas = set()
for s in filtradas["registro"]:
    if s.seq not in vistas:
        vistas.add(s.seq)
        seqs_unicas.append(s)
print(f"Secuencias únicas: {len(seqs_unicas)}") 

Secuencias únicas: 24


### DEFECTUOSAS

In [39]:
with open("../datosConfiguracion.json", "r") as config_file:
    config = json.load(config_file)
        
email = config["email"]
handle = Entrez.efetch(db="nucleotide", id=seqs_unicas[0].id, rettype="gb", retmode="text")
record = SeqIO.read(handle, "genbank")
handle.close()
for feature in record.features:
    print(feature)

type: source
location: [0:1793](+)
qualifiers:
    Key: collection_date, Value: ['2016']
    Key: db_xref, Value: ['taxon:1980456']
    Key: geo_loc_name, Value: ['Switzerland']
    Key: host, Value: ['Homo sapiens']
    Key: isolate, Value: ['ANDV-CH-LS-2016']
    Key: mol_type, Value: ['viral cRNA']
    Key: organism, Value: ['Orthohantavirus andesense']
    Key: segment, Value: ['S']

type: CDS
location: [11:1298](+)
qualifiers:
    Key: codon_start, Value: ['1']
    Key: product, Value: ['nucleocapsid protein']
    Key: protein_id, Value: ['WNM89964.1']
    Key: translation, Value: ['MSTLQELQENITAHEQQLVTARQKLKDAEKAVEVDPDDVNKSTLQSRRAAVSTLETKLGELKRQLADLVAAQKLATKPVDPTGLEPDDHLKEKSSLRYGNVLDVNSIDLEEPSGQTADWKAIGAYILGFAIPIILKALYMLSTRGRQTVKDNKGTRIRFKDDSSFEEVNGIRKPKHLYVSMPTAQSTMKAEEITPGRFRTIACGLFPAQVKARNIISPVMGVIGFGFFVKDWMDRIEEFLAAECPFLPKPKVASEAFMSTNKMYFLNRQRQVNESKVQDIIDLIDHAETESATLFTEIATPHSVWVFACAPDRCPPTALYVAGVPELGAFFSILQDMRNTIMASKSVGTAEEKLKKKSAFYQSYLRRTQSMGIQLDQKIIILYMLSWGKEAVNHFHLGDDMDPEL

In [ ]:
for s in seqs_unicas:
    s.seq.translate()

10
